In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
import joblib
import os

from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.svm import LinearSVC, SVC
from sklearn.linear_model import LogisticRegression
from sklearn.naive_bayes import MultinomialNB
from sklearn.ensemble import RandomForestClassifier
from xgboost import XGBClassifier
from sklearn.preprocessing import LabelEncoder
from sklearn.metrics import (
    accuracy_score,
    classification_report,
    confusion_matrix,
    ConfusionMatrixDisplay
)
from sklearn.pipeline import Pipeline
from sklearn.calibration import CalibratedClassifierCV

In [ ]:
from datasets import load_dataset

ds = load_dataset("papluca/language-identification")

README.md: 0.00B [00:00, ?B/s]

train.csv:   0%|          | 0.00/12.0M [00:00<?, ?B/s]

valid.csv: 0.00B [00:00, ?B/s]

test.csv: 0.00B [00:00, ?B/s]

Generating train split:   0%|          | 0/70000 [00:00<?, ? examples/s]

Generating validation split:   0%|          | 0/10000 [00:00<?, ? examples/s]

Generating test split:   0%|          | 0/10000 [00:00<?, ? examples/s]

In [ ]:
ds

DatasetDict({
    train: Dataset({
        features: ['labels', 'text'],
        num_rows: 70000
    })
    validation: Dataset({
        features: ['labels', 'text'],
        num_rows: 10000
    })
    test: Dataset({
        features: ['labels', 'text'],
        num_rows: 10000
    })
})

output labels `languages`:<br>
arabic (ar), bulgarian (bg), german (de), modern greek (el), english (en), spanish (es), french (fr), hindi (hi), italian (it), japanese (ja), dutch (nl), polish (pl), portuguese (pt), russian (ru), swahili (sw), thai (th), turkish (tr), urdu (ur), vietnamese (vi), and chinese (zh)

In [ ]:

# ─────────────────────────────────────────────
# 1. LOAD DATA
# ─────────────────────────────────────────────

train_df = ds['train'].to_pandas()
val_df = ds['validation'].to_pandas()
test_df = ds['test'].to_pandas()

print("Train size :", len(train_df))
print("Val size   :", len(val_df))
print("Test size  :", len(test_df))
print("Languages  :", sorted(train_df["labels"].unique()))
print()

Train size : 70000
Val size   : 10000
Test size  : 10000
Languages  : ['ar', 'bg', 'de', 'el', 'en', 'es', 'fr', 'hi', 'it', 'ja', 'nl', 'pl', 'pt', 'ru', 'sw', 'th', 'tr', 'ur', 'vi', 'zh']



In [ ]:

# ─────────────────────────────────────────────
# 2. MINIMAL PREPROCESSING
# ─────────────────────────────────────────────
import re

def preprocess(text: str) -> str:
    if not isinstance(text, str):
        return ""
    text = text.encode("utf-8", errors="ignore").decode("utf-8")
    # text = re.sub(r"http\S+|www\.\S+", " ", text)
    # text = re.sub(r"[@#]\S+", " ", text)
    # text = re.sub(r"\s+", " ", text).strip()
    text = text.lower()
    return text

train_df["clean"] = train_df["text"].apply(preprocess)
val_df["clean"]   = val_df["text"].apply(preprocess)
test_df["clean"]  = test_df["text"].apply(preprocess)

X_train, y_train = train_df["clean"], train_df["labels"]
X_val,   y_val   = val_df["clean"],   val_df["labels"]
X_test,  y_test  = test_df["clean"],  test_df["labels"]


In [ ]:
# Encoder for XGBoost (requires numeric labels)
le = LabelEncoder()
y_train_num = le.fit_transform(y_train)
y_val_num = le.transform(y_val)

In [ ]:
# ─────────────────────────────────────────────
# 3. FEATURE EXTRACTION
# ─────────────────────────────────────────────
vectorizer = TfidfVectorizer(
    analyzer="char_wb",
    ngram_range=(2, 4),
    min_df=3,
    sublinear_tf=True,
    lowercase=False
)

In [ ]:
# ─────────────────────────────────────────────
# 4. MODELS
# ─────────────────────────────────────────────
models = {
    # "Logistic Regression": Pipeline([
    #     ("tfidf", vectorizer),
    #     ("clf", LogisticRegression(
    #         max_iter=1000,
    #         C=1.0,
    #         penalty='l2',
    #         solver='lbfgs',
    #         n_jobs=-1,
    #         random_state=42
    #     ))
    # ]),

    "LinearSVC (calibrated)": Pipeline([
        ("tfidf", vectorizer),
        ("clf", CalibratedClassifierCV(
            LinearSVC(
                max_iter=2000,
                C=0.5,
                penalty='l2',
                loss='squared_hinge',
                dual=False
            ),
            method='sigmoid',
            cv=5
        ))
    ]),

    # "Random Forest": Pipeline([
    #     ("tfidf", vectorizer),
    #     ("clf", RandomForestClassifier(
    #         n_estimators=300,
    #         max_depth=25,
    #         min_samples_split=5,
    #         min_samples_leaf=2,
    #         max_features='sqrt',
    #         bootstrap=True,
    #         n_jobs=-1,
    #         random_state=42
    #     ))
    # ]),

    # "XGBoost": Pipeline([
    #     ("tfidf", vectorizer),
    #     ("clf", XGBClassifier(
    #         n_estimators=300,
    #         max_depth=6,
    #         learning_rate=0.05,
    #         subsample=0.8,
    #         colsample_bytree=0.8,
    #         reg_alpha=0.1,
    #         reg_lambda=1.0,
    #         min_child_weight=3,
    #         gamma=0.1,
    #         n_jobs=-1,
    #         random_state=42,
    #         eval_metric='mlogloss'
    #     ))
    # ]),

    # "Multinomial NB": Pipeline([
    #     ("tfidf", vectorizer),
    #     ("clf", MultinomialNB(
    #         alpha=0.1
    #     ))
    # ]),
}

In [ ]:
# ─────────────────────────────────────────────
# 5. TRAIN + VALIDATE
# ─────────────────────────────────────────────
print("=" * 55)
print("Validation results")
print("=" * 55)

val_results = {}
for name, pipeline in models.items():
    # Handle XGBoost numeric labels
    current_y_train = y_train_num if "XGBoost" in name else y_train
    current_y_val = y_val_num if "XGBoost" in name else y_val

    pipeline.fit(X_train, current_y_train)
    val_preds = pipeline.predict(X_val)
    acc = accuracy_score(current_y_val, val_preds)
    val_results[name] = acc
    print(f"{name:<35} Val Accuracy: {acc:.4f}")

best_model_name = max(val_results, key=val_results.get)
print(f"\nBest model on validation set: {best_model_name}")



Validation results
LinearSVC (calibrated)              Val Accuracy: 0.9955

Best model on validation set: LinearSVC (calibrated)


In [ ]:

# ─────────────────────────────────────────────
# 6. EVALUATE BEST MODEL ON TEST SET
# ─────────────────────────────────────────────
best_pipeline = models[best_model_name]   # already fitted above

test_preds = best_pipeline.predict(X_test)
test_acc   = accuracy_score(y_test, test_preds)

print("\n" + "=" * 55)
print("TEST SET RESULTS  —  " + best_model_name)
print("=" * 55)
print(f"Accuracy : {test_acc:.4f}\n")
print(classification_report(y_test, test_preds))


TEST SET RESULTS  —  LinearSVC (calibrated)
Accuracy : 0.9957

              precision    recall  f1-score   support

          ar       1.00      1.00      1.00       500
          bg       1.00      1.00      1.00       500
          de       1.00      1.00      1.00       500
          el       1.00      1.00      1.00       500
          en       1.00      1.00      1.00       500
          es       1.00      1.00      1.00       500
          fr       1.00      1.00      1.00       500
          hi       1.00      0.97      0.98       500
          it       1.00      1.00      1.00       500
          ja       1.00      1.00      1.00       500
          nl       1.00      1.00      1.00       500
          pl       1.00      1.00      1.00       500
          pt       1.00      1.00      1.00       500
          ru       1.00      1.00      1.00       500
          sw       0.94      1.00      0.97       500
          th       1.00      1.00      1.00       500
          tr     

In [ ]:
# ─────────────────────────────────────────────
# 7. CONFIDENCE-THRESHOLD INFERENCE
# ─────────────────────────────────────────────

# If the model's top probability is below THRESHOLD, return "uncertain"

THRESHOLD = 0.60

def predict_language(text: str, threshold: float = THRESHOLD) -> dict:
    """
    Returns:
        {
            "language"  : predicted language or "uncertain",
            "confidence": float,
            "reliable"  : bool
        }
    """
    clean = preprocess(text)
    proba = best_pipeline.predict_proba([clean])[0]
    classes = best_pipeline.classes_
    top_idx  = np.argmax(proba)
    top_conf = proba[top_idx]
    top_lang = classes[top_idx]

    if top_conf < threshold:
        return {"language": "uncertain", "confidence": float(top_conf), "reliable": False}
    return {"language": top_lang, "confidence": float(top_conf), "reliable": True}


it the model was uncertain then we need to use llm prompting (specially is the dataset is a bit far from the real production dataset it doesn't have emojis, the language is formal) so accuracy of the model may drop

In [ ]:
# ─────────────────────────────────────────────
# 9. SAVE THE BEST MODEL
# ─────────────────────────────────────────────
os.makedirs("models", exist_ok=True)
joblib.dump(best_pipeline, "models/language_detector.pkl")
print("\nModel saved → models/language_detector.pkl")


Model saved → models/language_detector.pkl


that model will be used in production <br>
as this notebook just for development

In [ ]:
file_path = "language_detector.pkl"
file_size_bytes = os.path.getsize(file_path)
file_size_mb = file_size_bytes / (1024 * 1024)
print(f"The size of '{file_path}' is: {file_size_mb:.2f} MB")

The size of 'language_detector.pkl' is: 193.12 MB


# TEST

In [ ]:
text ="شغل عالي من الملك"

In [ ]:
predict_language(text= text)

{'language': 'ar', 'confidence': 0.9927036304675985, 'reliable': True}